# 🔍 FastAPI Foundry — RAG Builder (Google Colab)

Этот notebook позволяет:
1. **Установить** зависимости (FAISS, sentence-transformers)
2. **Загрузить** документы (вручную, Google Drive или GitHub)
3. **Построить** FAISS-индекс с эмбеддингами
4. **Протестировать** поиск по индексу
5. **Скачать** готовый индекс для FastAPI Foundry

> **Совет:** Runtime → Change runtime type → **GPU (T4)** для ускорения

## 1. Установка зависимостей

In [ ]:
!pip install -q sentence-transformers faiss-cpu numpy

## 2. Конфигурация

In [ ]:
import json, hashlib, logging, zipfile, os
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional
from collections import Counter

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# --- Settings (change as needed) ---
MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'  # 768-dim, best quality
# MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'  # 384-dim, faster
CHUNK_SIZE = 1000
OVERLAP    = 50
DOCS_DIR   = Path('/content/docs')
OUTPUT_DIR = Path('/content/rag_index')
SUPPORTED  = {'.md', '.txt', '.html', '.rst'}

DOCS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Config ready')

## 3. Загрузка документов
Выберите **один** вариант.

In [ ]:
# Option A: Upload files manually
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    (DOCS_DIR / name).write_bytes(data)
print(f'✅ Uploaded {len(uploaded)} file(s)')

In [ ]:
# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DOCS_DIR = Path('/content/drive/MyDrive/my_docs')

In [ ]:
# Option C: Clone GitHub repo
# !git clone https://github.com/hypo69/FastApiFoundry-Docker /content/repo
# DOCS_DIR = Path('/content/repo/docs')

In [ ]:
found = [p for p in DOCS_DIR.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED]
print(f'📄 Found {len(found)} document(s):')
for f in found[:20]:
    print(f'  {f.relative_to(DOCS_DIR)}')
if len(found) > 20:
    print(f'  ... and {len(found)-20} more')

## 4. RAG Indexer

In [ ]:
class RAGIndexer:
    """Build a FAISS index from a directory of documents."""

    def __init__(self, model_name: str = MODEL_NAME) -> None:
        self.model_name = model_name
        self.model = None
        self.chunks: List[Dict[str, Any]] = []
        self.embeddings = None
        self.docs_root: Optional[Path] = None

    def load_model(self) -> None:
        logger.info(f'Loading model: {self.model_name}')
        self.model = SentenceTransformer(self.model_name)
        logger.info('✅ Model loaded')

    def _read(self, path: Path) -> str:
        try:
            return path.read_text(encoding='utf-8')
        except Exception as e:
            logger.warning(f'⚠️ Cannot read {path}: {e}')
            return ''

    def chunk_text(self, text: str) -> List[str]:
        if len(text) <= CHUNK_SIZE:
            return [text]
        chunks, start = [], 0
        while start < len(text):
            end = start + CHUNK_SIZE
            if end < len(text):
                for i in range(end, max(start + CHUNK_SIZE // 2, end - 100), -1):
                    if text[i] in '.!?':
                        end = i + 1
                        break
            chunk = text[start:end].strip()
            if chunk:
                chunks.append(chunk)
            start = end - OVERLAP
        return chunks

    def _process_md(self, content: str, meta: Dict) -> List[Dict]:
        chunks, section, lines = [], 'Introduction', []
        for line in content.split('\n'):
            line = line.strip()
            if line.startswith('#'):
                if lines:
                    text = '\n'.join(lines).strip()
                    if text:
                        for c in self.chunk_text(text):
                            chunks.append({**meta, 'section': section, 'text': c})
                section, lines = line.lstrip('#').strip(), []
            elif line:
                lines.append(line)
        if lines:
            text = '\n'.join(lines).strip()
            if text:
                for c in self.chunk_text(text):
                    chunks.append({**meta, 'section': section, 'text': c})
        return chunks

    def index_directory(self, docs_dir: Path) -> None:
        logger.info(f'Indexing: {docs_dir}')
        if not docs_dir.exists():
            logger.error(f'❌ Not found: {docs_dir}')
            return
        self.docs_root = docs_dir
        count = 0
        for path in docs_dir.rglob('*'):
            if not path.is_file() or path.suffix.lower() not in SUPPORTED:
                continue
            content = self._read(path)
            if not content:
                continue
            rel = str(path.relative_to(docs_dir))
            meta = {'source': path.name, 'path': rel,
                    'checksum': hashlib.md5(content.encode()).hexdigest()}
            if path.suffix.lower() == '.md':
                self.chunks.extend(self._process_md(content, meta))
            else:
                for c in self.chunk_text(content):
                    self.chunks.append({**meta, 'section': 'Content', 'text': c})
            count += 1
        logger.info(f'✅ Files: {count}, chunks: {len(self.chunks)}')

    def create_embeddings(self) -> None:
        if not self.chunks:
            raise ValueError('No chunks — run index_directory() first')
        logger.info(f'Encoding {len(self.chunks)} chunks…')
        texts = [c['text'] for c in self.chunks]
        self.embeddings = self.model.encode(
            texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True
        ).astype('float32')
        logger.info(f'✅ Shape: {self.embeddings.shape}')

    def save_index(self, output_dir: Path) -> None:
        output_dir.mkdir(parents=True, exist_ok=True)
        faiss.normalize_L2(self.embeddings)
        index = faiss.IndexFlatIP(self.embeddings.shape[1])
        index.add(self.embeddings)
        faiss.write_index(index, str(output_dir / 'faiss.index'))
        (output_dir / 'chunks.json').write_text(
            json.dumps(self.chunks, ensure_ascii=False, indent=2), encoding='utf-8')
        (output_dir / 'index_info.json').write_text(
            json.dumps({'model': self.model_name, 'chunks_count': len(self.chunks),
                        'dimension': self.embeddings.shape[1],
                        'created_at': datetime.now().isoformat(), 'version': '1.0.0'},
                       ensure_ascii=False, indent=2), encoding='utf-8')
        logger.info(f'✅ Saved to {output_dir} ({index.ntotal} vectors)')

print('✅ RAGIndexer ready')

## 5. Построение индекса

In [ ]:
indexer = RAGIndexer()
indexer.load_model()
indexer.index_directory(DOCS_DIR)

if not indexer.chunks:
    print('❌ No documents found. Add files to DOCS_DIR and re-run.')
else:
    indexer.create_embeddings()
    indexer.save_index(OUTPUT_DIR)
    print(f'\n🎉 Done! Index saved to {OUTPUT_DIR}')

## 6. Тест поиска

In [ ]:
class RAGSearch:
    """Search the FAISS index."""

    def __init__(self, index_dir: Path, model_name: str = MODEL_NAME) -> None:
        self.model  = SentenceTransformer(model_name)
        self.index  = faiss.read_index(str(index_dir / 'faiss.index'))
        with open(index_dir / 'chunks.json', encoding='utf-8') as f:
            self.chunks = json.load(f)
        print(f'✅ Index: {self.index.ntotal} vectors, {len(self.chunks)} chunks')

    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        vec = self.model.encode([query]).astype('float32')
        faiss.normalize_L2(vec)
        scores, indices = self.index.search(vec, top_k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < len(self.chunks):
                chunk = self.chunks[idx].copy()
                chunk['score'] = round(float(score), 4)
                results.append(chunk)
        return results

    def show(self, query: str, top_k: int = 3) -> None:
        print(f'\n🔍 "{query}"\n' + '─'*60)
        for i, r in enumerate(self.search(query, top_k), 1):
            print(f'[{i}] score={r["score"]} | {r["path"]} § {r["section"]}')
            print(r['text'][:300])
            print()

searcher = RAGSearch(OUTPUT_DIR)

# Change queries to match your documents
searcher.show('installation')
searcher.show('API endpoints')
searcher.show('configuration')

## 7. Интерактивный поиск

In [ ]:
query = input('Search query: ')
top_k = int(input('Results count [5]: ') or 5)
searcher.show(query, top_k)

## 8. Статистика

In [ ]:
info = json.loads((OUTPUT_DIR / 'index_info.json').read_text())
print('📊 Index info:')
for k, v in info.items():
    print(f'  {k}: {v}')

lengths = [len(c['text']) for c in searcher.chunks]
print(f'\n📏 Chunk lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}')

print('\n📄 Top sources:')
for src, cnt in Counter(c['source'] for c in searcher.chunks).most_common(10):
    print(f'  {src}: {cnt} chunks')

## 9. Скачать индекс

Распакуйте `rag_index.zip` в папку `rag_index/` проекта FastAPI Foundry.

In [ ]:
zip_path = '/content/rag_index.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR.iterdir():
        zf.write(f, f.name)

size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'📦 rag_index.zip — {size_mb:.1f} MB')

from google.colab import files
files.download(zip_path)

## 10. Сохранить в Google Drive (опционально)

In [ ]:
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# shutil.copy(zip_path, '/content/drive/MyDrive/rag_index.zip')
# print('✅ Saved to Google Drive')

---
## Использование в FastAPI Foundry

1. Распакуйте `rag_index.zip` → папка `rag_index/` в корне проекта
2. В `config.json`:
```json
{
  "rag_system": {
    "enabled": true,
    "index_dir": "./rag_index"
  }
}
```
3. `python run.py`
4. Проверка: `GET http://localhost:9696/api/v1/rag/status`